# 03 — Quality, and the two gates that are not the same gate

Notebook 02 kept saying "the gate refused it". This one opens the gates.

There are **two**, they answer different questions, and confusing them is the
easy mistake:

| | question | what happens to the refused |
|---|---|---|
| **acceptance** (`quality/gate.py`) | is this text good enough to *stop* the cascade? | it **stays in the contest** — it may be the best reading there is |
| **replacement** (`quality/rejection.py`) | is it better than what I already had, and did it lose nothing? | it is **discarded** |

The first compares against a threshold; the second compares against a concrete
text and has already concluded the new one is worse. Letting a replacement-gate
refusal compete would cancel the gate, because volume is usually on the wrong
side: the corrupted text is precisely the longest one (CLAUDE.md §4).

There is exactly **one** acceptance criterion in the pipeline (§2). Whoever
decides the current step solved it and whoever decides the next one is worth
paying for call the same function. Two competing notions of "adequate
extraction" in one pipeline — the step approving itself by one criterion and the
cascade refusing it by another — is the defect `evaluate` exists to stop
repeating.

In [ ]:
import logging

logging.disable(logging.INFO)

from autosxtract.pdf.profile import PageProfile
from autosxtract.quality.gate import evaluate
from autosxtract.quality.metrics import glyph_index_ratio
from autosxtract.quality.scoring import score_text
from autosxtract.quality.stamp import default as stamp_for

SHEET = PageProfile(pages=1, has_image=True)      # ink on the page
BLANK = PageProfile(pages=1)                       # nothing drawn at all

BANNER = (
    "Este documento e copia do original assinado digitalmente por FULANO DE "
    "TAL. Para conferir o original acesse o site do tribunal e informe o "
    "codigo de verificacao 8A2F91C. fls. 42\n"
)
PROSE = (
    "O requerente vem respeitosamente a presenca de Vossa Excelencia nos "
    "autos do processo 0001234-56.2020.8.12.0001 requerer a citacao do "
    "requerido, tendo em vista a decisao proferida pela vara civel e a "
    "certidao do oficial de justica que instrui o presente pedido. "
)
print(len(BANNER), "characters of banner |", len(PROSE), "characters of prose")

## The acceptance gate: four questions, in this order

1. **Is there anything to read?** A page with no visual content never escalates
   — a blank back page has no text to recover and the expensive step is pure
   cost. That question is what separated an archive's 4 legitimately empty pages
   from its 227 false successes.
2. **Did any word survive outside the stamp?** Below the floor, what came back is
   the conformity banner, not the document.
3. **Is it text, or a glyph index?**
4. **Does the density match the size of the sheet?**

No I/O, no network, no global configuration: text, page profile, thresholds. That
is what lets it be called from both sides of the decision at no cost.

In [ ]:
cases = [
    ("blank back page",   "",                                  BLANK, 0.0),
    ("only the stamp",    BANNER * 3,                          SHEET, 0.0),
    ("glyph index",       PROSE + "g40g86g87g72g14g99g23g81 " * 20, SHEET, None),
    ("one short line",    "Intimacao cumprida em 17/03/2005.", SHEET, 0.0),
    ("a real reading",    PROSE * 3,                           SHEET, 0.0),
]

print(f"{'case':<18} {'chars':>6} {'escalate':<10} reason")
print("-" * 78)
for label, text, profile, _ in cases:
    verdict = evaluate(text, profile, glyph_index=glyph_index_ratio(text))
    print(f"{label:<18} {len(text):>6} {str(verdict.escalate):<10} {verdict.reason}")

`escalate=False` on the blank page is not "this text is good": it is "there is
nothing here to recover, and paying for OCR would buy nothing". The reason
travels into the provenance, which is why the verdict carries a sentence rather
than a bool. A gate returning a bare `bool` would satisfy the cascade and destroy
the audit trail.

## The stamp, and why it comes off before anything is measured

Every digital case-file system prints a conformity banner in the margin, in a
font whose encoding **survives when the body of the page produces nothing**. That
is 250 to 600 characters that sail past any size threshold. In an audit of 1,339
documents, **227 extractions looked successful and all there was, was the
stamp**.

So: measuring an extraction without stripping the stamp is measuring the stamp.
Here is the same text put through the same gate twice — once with the stripper in
place, once with a pattern that matches nothing.

In [ ]:
stamp_only = BANNER * 3
MATCHES_NOTHING = ("(?!x)x",)

stripper = stamp_for()
print("raw characters          :", len(stamp_only))
print("characters after strip  :", len(stripper.strip(stamp_only)))
print("useful words outside it :", stripper.count(stamp_only))
print()
print("measured WITH stripping   :", evaluate(stamp_only, SHEET))
print("measured WITHOUT stripping:", evaluate(stamp_only, SHEET, stamps=MATCHES_NOTHING))

Read those last two lines against each other. The identical 534 characters are
"adequate extraction" to a pipeline that does not strip, and "only 0 useful words
outside the stamp" to one that does. Nothing about the document changed; the
measurement did. 227 documents were on the wrong side of that line.

This is also the library's **adaptation seam**. The shipped patterns are
Brazilian court boilerplate; another corpus supplies its own through
`Config.stamps` or a pattern pack, and no measurement code changes anywhere,
because everyone measures through the same `StampStripper`. Notebook 07 writes
one.

## The score

The score is a sum of penalties against 1.0, and **every penalty carries its
reason in words** — the number alone is not auditable. Three families:
form (odd characters, short lines, fragmentation), plausibility (glyph index,
broken encoding, no function words at all) and domain vocabulary, which is the
only one that can *add*.

In [ ]:
for label, text in [
    ("a real reading", PROSE * 3),
    ("junk",           "rn cl 1i vv xzq " * 45),
    ("glyph index",    "g40g86g87g72g14g99 " * 40),
    ("empty",          ""),
]:
    assessment = score_text(text)
    print(f"{label:<15} score {assessment['score']:<7} {assessment['label']}")
    for reason in assessment["reasons"]:
        print(f"{'':<15}   - {reason}")
    print()

Empty text scores `0.0` **explicitly**, and that line of code is load-bearing:
summing the three degenerate-text penalties leaves an empty string at 0.15, which
is enough to compete with real text in the contest of notebook 02.

Note what the junk did *not* trigger: engine confidence never appears here.
Measured on 60 documents audited by four reviewers, engine confidence does not
separate a good reading from an unsafe one — there was an unsafe document at
confidence 100 (§7). It enters the pipeline as a floor against degenerate output
and never as a criterion.

## The replacement gate: refused means discarded

This one runs only after an `expensive` step, and it compares against a concrete
earlier text rather than a threshold. The naive rule — `len(new) > len(previous)`
— gets both of the costliest cases wrong:

- **coverage**: a document transcribed up to page 10 of 15 is longer than a bad
  extraction of all 15, and silently replaced it. That is how a power of attorney
  lost three notarial acts.
- **fidelity**: the expensive step rewrites with better layout and corrupts
  digits the previous step had read correctly. Length and text score are blind to
  it — `9XXYZ3ZE...` scores exactly like `9XXYZ32E...`.

What catches the second is comparing the **sets** of verifiable tokens before and
after. Set comparison, not positional, because a new step legitimately reorders
the layout.

In [ ]:
from autosxtract.quality.anchors import anchors, lost
from autosxtract.quality.rejection import assess_replacement

PREVIOUS = (
    "Certidao expedida nos autos do processo 0001234-56.2020.8.12.0001, "
    "protocolo 882167, referente ao veiculo de chassi 9XXYZ32E41A099887, em "
    "que o oficial de justica certificou que a diligencia foi cumprida na "
    "data de 17/03/2005 conforme determinado pela vara civel."
)
CORRUPTED = PREVIOUS.replace("882167", "882187").replace("32E", "3ZE") + (
    " Segue a transcricao integral do documento com as demais informacoes "
    "constantes dos autos, lidas pela etapa mais cara desta cascata."
)
HONEST = PREVIOUS + (
    " Segue a transcricao integral do documento com as demais informacoes "
    "constantes dos autos, sem corromper digito algum da leitura anterior."
)

print("anchors in the previous text:", sorted(anchors(PREVIOUS)))
print("anchors the new text lost   :", sorted(lost(PREVIOUS, CORRUPTED)))
print()

trials = [
    ("longer, two digits changed", CORRUPTED, PREVIOUS, 1, 1, 1),
    ("longer, nothing lost",       HONEST,    PREVIOUS, 1, 1, 1),
    ("page ceiling cut it short",  HONEST,    PREVIOUS, 15, 10, 10),
    ("previous was degenerate",    CORRUPTED, "sem texto util", 1, 1, 1),
]
for label, new, old, pages, sent, answered in trials:
    verdict = assess_replacement(
        new, old, document_pages=pages, pages_sent=sent, pages_answered=answered
    )
    print(f"{label:<28} accepted={str(verdict.accepted):<6} {verdict.reason or ''}")

The last row is the exemption that keeps the gate from becoming a no-op against
itself: the gate is only valid while the previous text is a **trustworthy
reference**. When the previous step collapsed and read almost nothing, there is
nothing to check against and partial text beats nothing — so the digit-corrupted
reading is accepted, because refusing it would reject exactly the document the
expensive step exists to rescue.

The third row is the page ceiling: refusing outright would trade a rich,
incomplete text for a poor, complete one, so the density comparison decides — and
when it accepts a truncation it does so with a **warning** in the result, never in
silence. Without that warning the power of attorney lost its three acts leaving no
trace.

## Consensus and agreement: two engines, two different questions

No pixel statistic tells a blank page from a dense but faded one. Nine families
were tried — total ink, projection bands, compressed bytes per page, image
coverage, six preprocessing tracks, the CCpdf born-digital rule — and all failed
for the same reason: the two cases produce identical statistics.

What separates them is measuring instead of estimating, with engines of
**different architectures**, because errors from independent models do not
correlate. Measured on a real archive at the pipeline's floor of 12 useful words,
the separation was absolute: 0–4 useful words against 49–118, with no grey zone.

In [ ]:
from autosxtract.quality.consensus import assess_agreement, assess_emptiness

for label, readings in [
    ("all three see nothing", {"engine_a": 0, "engine_b": 1, "engine_c": 2}),
    ("one dissenter",         {"engine_a": 0, "engine_b": 1, "engine_c": 40}),
    ("only one voted",        {"engine_a": 0}),
]:
    verdict = assess_emptiness(readings, word_floor=12)
    print(f"{label:<24} empty={str(verdict.empty):<6} {verdict.evidence}")

The asymmetry is deliberate: **one dissenting engine is enough not to declare the
page empty**. Declaring a page with text empty discards information; the reverse
only wastes time. And a single vote is never a consensus — "I could not read it"
is not "there is nothing here".

The agreement gate inverts the question. Emptiness proves *absence of content*;
agreement proves the reading is **complete** — that the page is short because the
document is short, and not because the extraction failed. Measured on 935
documents: 23 vetoes, the expensive step fell from 25 calls to 16, and real
content went **up** by 424 characters. It costs nothing, because at the point of
decision the cascade already holds both readings.

In [ ]:
second_reading = PROSE.replace("citacao", "cltacao").replace("justica", "justiga")

agreement = assess_agreement(
    {"engine_a": PROSE * 3, "engine_b": second_reading * 3},
    word_floor=12,
    min_similarity=0.60,
)
print("agree      :", agreement.agree)
print("similarity :", round(agreement.similarity, 3))
print("evidence   :", agreement.evidence)

The measure is Jaccard over the vocabulary **outside the stamp** — insensitive to
order and repetition, so two engines breaking lines differently does not count as
disagreement. The 0.60 threshold was calibrated on 24 real escalations: 13 landed
between 0.00 and 0.48 and 11 between 0.65 and 0.80, and the threshold sits in the
empty gap between the two clusters.

## Containment layers: what an engine with geometry buys

The premise comes from an audit rather than intuition. A small OCR engine reads
the *body* of a document almost perfectly — case number, parties, address, dates,
amounts — and the error concentrates in four places: a vertical stamp read
sideways, a signature over printed text, a scrambled two-column header, and
run-together capitals. No OCR reads *through* a stamp. So the strategy is not
"read better", it is **contain the damage, flag where it is, recover what is
recoverable**.

Measured on 895 pages against the same engine without the layers: entity recall
0.902 → 0.921, median CER 0.132 → 0.129, and p50 latency **falls** from 298 to
236 ms — because the ceiling on re-reads costs less than the junk lines that stop
reaching the recogniser. 79 pages improve, 4 get worse (all already bad), clean
pages are untouched.

This is also where the engine contract's optional half earns itself (§15):
`read_page` returns lines with polygons and scores, and `None` is the honest
answer from an engine without geometry. Here is a page assembled by hand, so the
classification is visible line by line.

In [ ]:
from autosxtract.quality.lines import contain
from autosxtract.types import Line, Page


def box(x1, y1, x2, y2):
    return ((x1, y1), (x2, y1), (x2, y2), (x1, y2))


page = Page(
    width=1240,
    height=1754,
    lines=[
        Line("EXCELENTISSIMO SENHOR DOUTOR JUIZ DE DIREITO DA VARA CIVEL", 0.96, box(120, 120, 1100, 160)),
        Line("O requerente, nos autos do processo 0001234-56.2020.8.12.0001, vem requerer", 0.95, box(120, 200, 1100, 240)),
        Line("a citacao do requerido conforme decisao proferida por esta vara civel.", 0.94, box(120, 250, 1100, 290)),
        Line("PROTOCOLO 8A2F91C ASSINADO DIGITALMENTE", 0.71, box(30, 300, 80, 1400)),   # tall and narrow
        Line("xzq lqp rn cl vv 1i", 0.31, box(120, 900, 700, 940)),
        Line("ESTADODEMATOGROSSODOSUL", 0.88, box(120, 1100, 700, 1140)),
    ],
)

contained = contain(page)
for line in contained.lines:
    print(f"  {line.kind:<10} coverage={line.coverage:.2f}  {line.text[:52]!r}")

print()
print(contained.text)
print()
print(contained.report())

The vertical line was recognised by its shape — tall and narrow — before anything
read it, and left the body of the text. The junk line became a marker instead of
polluting the page. `trusted_fraction` and `suggested_action` are Layer 3: the
module **measures and does not decide**, and the policy belongs to the caller.

The principle behind the thresholds is worth more than the thresholds: `illegible`
only for **unambiguous** junk. Anything doubtful — a name misread under a
signature, a blurred header — becomes `suspect`, so the text passes and the page
merely loses confidence. That way the weak signal is not thrown away.

Now the same thing on a real engine, if this machine has one with geometry.

In [ ]:
import pymupdf

from autosxtract.engines import available, get
from autosxtract.pdf.render import render


def scanned_pdf(text: str, dpi: int = 180) -> bytes:
    doc = pymupdf.open()
    doc.new_page().insert_textbox(pymupdf.Rect(50, 50, 550, 380), text, fontsize=11)
    digital = doc.tobytes()
    doc.close()
    source = pymupdf.open("pdf", digital)
    out = pymupdf.open()
    pixmap = source[0].get_pixmap(dpi=dpi, colorspace=pymupdf.csGRAY)
    sheet = out.new_page(width=source[0].rect.width, height=source[0].rect.height)
    sheet.insert_image(sheet.rect, stream=pixmap.tobytes("png"))
    data = out.tobytes()
    out.close()
    source.close()
    return data


ready = [info.name for info in available()]
if not ready:
    print("no OCR engine here — the hand-built page above is the whole lesson.")
    print("On a machine with one, this cell prints the same report over real pixels.")
else:
    engine = get(ready[0])
    images = render(scanned_pdf(PROSE * 2), dpi=150, max_pages=1)
    real_page = engine.read_page(images[0])
    if real_page is None:
        # The honest answer from an engine without geometry. It blocks nothing:
        # the cascade falls back to the plain string contract and records that
        # the layers did not run.
        print(f"{engine.name} exposes no line geometry — layers skipped, extraction unaffected")
    else:
        print(f"{engine.name}: {len(real_page.lines)} lines, "
              f"mean confidence {real_page.mean_confidence}")
        for line in real_page.lines[:4]:
            print(f"  {line.score:.2f}  {line.bbox}  {line.text[:48]!r}")
        print()
        print(contain(real_page).report())

## What you now know

- **one** acceptance criterion, called from both sides of every decision;
- the stamp comes off *before* the measurement, and the same 534 characters
  change verdict depending on whether it did;
- the score explains itself in words, and empty text is 0.0 by an explicit line
  rather than by arithmetic;
- the replacement gate compares against a text, not a threshold, and what it
  refuses is gone;
- two engines answer two different questions — is it empty, is it complete — and
  neither is a pixel statistic;
- geometry is optional, and what it buys is containment.

Next: **04 — configuration**, and the reason a single isolated measurement in
this project has already lied.